# Módulo 3 — Expresión de la proteína de la cápside (coat protein)
### BruSafe / CAPSID · iGEM 2026

Este notebook reproduce las dos figuras del panel de M3:

- **(a)** la expresión de mRNA y de proteína hasta alcanzar el estado estacionario
- **(b)** la justificación de la aproximación *D = P/2*

Los resultados son **simulaciones del modelo con parámetros de la literatura**, no datos de
laboratorio. Corresponden a la predicción *a priori* (Modo A), es decir, a lo que se espera
observar antes de medir.

Las celdas se ejecutan en orden (`Runtime → Run all` en Colab).

## 1 · Preparación

In [ ]:
# ---------------------------------------------------------------
# Solo se usan numpy, scipy y matplotlib, que ya vienen instalados en Colab.
# No es necesario instalar nada.
# ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

RT = 0.593        # kcal/mol a ~298 K
N_CAP = 90        # dimeros de coat protein por capside T=3 (Ni 1995)

## 2 · Parámetros

Todos los valores provienen de la literatura sobre *E. coli* en general; **no corresponden a
nuestra cepa ni a nuestro plásmido**. Esos son precisamente los parámetros que deberá medir el
equipo de laboratorio.

In [ ]:
# Parametros: (valor central, (min, max), fuente)
# El rango no es un intervalo de confianza, sino el soporte del prior tomado de
# la literatura. Al cambiar los valores centrales las figuras se recalculan.
PARAMS = {
    "t_half_mrna_min":   (5.0,   (2.0, 8.0),   "Bernstein 2002; Selinger 2003"),
    "t_double_min":      (40.0,  (25., 60.),   "Milo 2010"),
    "alpha_tx_nM_min":   (6.0,   (1.0, 30.),   "Milo 2010"),
    "k_tl_permin":       (4.0,   (1.0, 12.),   "Milo 2010"),
    "delta_prot_permin": (0.002, (0.0, 0.01),  "Milo 2010"),
    "Kd_dim_uM":         (0.01,  (1e-3, 1.0),  "Ni 1995 (dimero obligado)"),
}

def central(params=PARAMS):
    """Extrae solo los valores centrales."""
    return {k: v[0] for k, v in params.items()}

p = central()
for k, (v, rng, src) in PARAMS.items():
    print(f"{k:<20} {v:<8} rango {str(rng):<14} {src}")

## 3 · El modelo

El modelo consta de dos ecuaciones diferenciales ordinarias acopladas (mRNA y proteína) y del
equilibrio de dimerización.

In [ ]:
def induction(t, t_on=30.0, ramp=10.0):
    """Interruptor suave del inductor (IPTG): 0 -> 1 alrededor de t_on."""
    return 1.0 / (1.0 + np.exp(-(t - t_on) / ramp))


def expression_ode(y, t, alpha, delta_m, mu, k_tl, delta_P, t_on):
    """dm/dt = alpha*u(t) - (delta_m + mu)*m
       dP/dt = 1e-3*k_tl*m - (delta_P + mu)*P

    m en nM, P en uM monomero-equivalente (1e-3 convierte nM -> uM).

    Nota: mu (dilución por crecimiento) aparece en ambas ecuaciones. Para la
    coat protein, que es estable (delta_P = 0.002 /min), mu domina la pérdida
    por un factor ~9: la vida útil de la proteína la fija la división celular
    y no la proteólisis.
    """
    m, P = y
    u = induction(t, t_on)
    dm = alpha * u - (delta_m + mu) * m
    dP = 1e-3 * k_tl * m - (delta_P + mu) * P
    return [dm, dP]


def derived_rates(p):
    """Convierte vidas medias en tasas de primer orden."""
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    return delta_m, mu


def simulate_expression(p, t_grid, t_on=30.0):
    """Integra las ODEs. Devuelve (m en nM, P en uM)."""
    delta_m, mu = derived_rates(p)
    sol = odeint(expression_ode, [0.0, 0.0], t_grid,
                 args=(p["alpha_tx_nM_min"], delta_m, mu,
                       p["k_tl_permin"], p["delta_prot_permin"], t_on),
                 rtol=1e-8, atol=1e-10)
    return sol[:, 0], sol[:, 1]


def expression_steady_state(p):
    """Forma cerrada con u -> 1:
         m_ss = alpha / (delta_m + mu)
         P_ss = 1e-3 * k_tl * m_ss / (delta_P + mu)
    Puede verificarse analíticamente."""
    delta_m, mu = derived_rates(p)
    m_ss = p["alpha_tx_nM_min"] / (delta_m + mu)
    P_ss = 1e-3 * p["k_tl_permin"] * m_ss / (p["delta_prot_permin"] + mu)
    return m_ss, P_ss


def monomer_dimer_split(P_tot_uM, Kd_dim_uM):
    """Reparto de equilibrio 2M <-> D:
         2D + M = P_tot ,  D = M^2 / Kd
    Resuelve (2/Kd)M^2 + M - P_tot = 0.

    Cuando Kd -> 0 (dímero fuerte, Ni 1995), D -> P_tot/2. Esta es la
    aproximación que usa el modelo principal; aquí se verifica en lugar de
    suponerse."""
    a = 2.0 / Kd_dim_uM
    M = (-1.0 + np.sqrt(1.0 + 4.0 * a * P_tot_uM)) / (2.0 * a)
    D = M * M / Kd_dim_uM
    return M, D


# --- comprobacion numerica ---
delta_m, mu = derived_rates(p)
m_ss, P_ss = expression_steady_state(p)
M_eq, D_eq = monomer_dimer_split(P_ss, p["Kd_dim_uM"])

print(f"delta_m = {delta_m:.4f} /min     mu = {mu:.5f} /min")
print(f"perdida de proteina: mu/delta_P = {mu/p['delta_prot_permin']:.1f}x  "
      f"(la dilucion domina)")
print(f"m_ss  = {m_ss:.1f} nM")
print(f"P_ss  = {P_ss:.2f} uM monomero-equivalente")
print(f"D_ss  = {D_eq:.2f} uM dimero  (fraccion dimerica = {2*D_eq/P_ss:.3f})")
print(f"P_ss/2 = {P_ss/2:.2f} uM  <- aproximacion de dimero perfecto")

## 4 · Las dos figuras del panel

El panel (a) integra las ecuaciones hasta 300 min. El panel (b) barre la constante de disociación
del dímero, $K_d$, para mostrar a partir de qué valor deja de ser válida la aproximación $D = P/2$.

In [ ]:
# Reproduce los dos paneles: (a) llegada al estado estacionario, (b) justificación de D = P/2
t = np.linspace(0, 300, 601)
m_t, P_t = simulate_expression(p, t, t_on=30.0)

# barrido de Kd para el panel (b), a P_tot fijo = P_ss
Kd_grid = np.logspace(-3, 1.5, 200)
frac_dimer = np.array([2.0 * monomer_dimer_split(P_ss, kd)[1] / P_ss
                       for kd in Kd_grid])

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.1))

# ---- panel a ----
axA.plot(t, m_t,       color="#c0392b", lw=1.9, label=r"mRNA $m$ (nM)")
axA.plot(t, P_t,       color="#17607a", lw=1.9, label=r"monómero $P$ (µM)")
axA.plot(t, P_t / 2.0, color="#d4a017", lw=1.7, ls="--",
         label=r"dímero $D = P/2$ (µM)")
axA.axvline(30.0, color="#8a9a95", lw=0.8, ls=":")
axA.annotate("inducción  $t_{on}$ = 30 min", xy=(30.0, 22.0),
             xytext=(115.0, 27.0), fontsize=7.5, color="#5b6b66",
             arrowprops=dict(arrowstyle="->", color="#8a9a95", lw=0.8))
axA.set_xlabel("tiempo (min)")
axA.set_ylabel("concentración")
axA.set_title("a   La expresión llega al estado estacionario", loc="left")
axA.legend(frameon=False, fontsize=8, loc="center right")

# ---- panel b ----
axB.plot(Kd_grid, frac_dimer, color="#17607a", lw=2.0)
axB.axvline(p["Kd_dim_uM"], color="#d4a017", lw=1.4, ls="--")
axB.text(p["Kd_dim_uM"] * 1.5, 0.55,
         f"Ni 1995: régimen de dímero fuerte\n"
         f"$K_d$ = {p['Kd_dim_uM']} µM → fracción {2*D_eq/P_ss:.3f}",
         fontsize=7.5, color="#7a5c00")
axB.set_xscale("log")
axB.set_xlabel(r"$K_d$ de dimerización (µM)")
axB.set_ylabel(r"fracción de proteína como dímero  $2D/P_{tot}$")
axB.set_title(r"b   Por qué se justifica $D = P/2$", loc="left")
axB.set_ylim(0.4, 1.02)

fig.tight_layout()
fig.savefig("M3_expresion.png", dpi=300, bbox_inches="tight")
plt.show()
print("guardado: M3_expresion.png")

## 5 · Barrido de temperatura

La consecuencia práctica de M3 es que reducir la temperatura disminuye el pico de dímero. Como el
óptimo de ensamblaje de M4 se encuentra en 2 µM, la temperatura de cultivo actúa como una **palanca
de control** sobre el rendimiento de ensamblaje y no solo como un efecto colateral.

In [ ]:
def scale_params_temperature(p, T_C, T_ref_C=37.0, Q10_growth=2.0, Q10_tx=2.0):
    """Reescala las tasas dependientes de temperatura con una ley Q10.

    Es una aproximación gruesa: permite predecir el signo y el orden de
    magnitud del efecto de bajar a 30 o 25 °C, pero no un valor preciso.

    Si se mide la curva de crecimiento a las tres temperaturas, Q10_growth
    se obtiene del dato y deja de ser un supuesto."""
    q = lambda Q10: Q10 ** ((T_C - T_ref_C) / 10.0)
    out = dict(p)
    out["t_double_min"]    = p["t_double_min"] / q(Q10_growth)
    out["alpha_tx_nM_min"] = p["alpha_tx_nM_min"] * q(Q10_tx)
    out["k_tl_permin"]     = p["k_tl_permin"] * q(Q10_tx)
    return out


t = np.linspace(0, 240, 481)
fig, ax = plt.subplots(figsize=(5.6, 4.0))
for T, c in [(37.0, "#c0392b"), (30.0, "#d4a017"), (25.0, "#17607a")]:
    _, P_T = simulate_expression(scale_params_temperature(p, T), t, t_on=30.0)
    D_T = P_T / 2.0
    ax.plot(t, D_T, color=c, lw=1.9, label=f"{T:.0f} °C")
    print(f"T = {T:4.1f} C -> D(240 min) = {D_T[-1]:5.2f} uM")

ax.axhline(2.0, color="#0b6e5a", lw=1.3, ls="--")
ax.text(236, 2.08, "óptimo de M4: 2 µM", ha="right", fontsize=7.5,
        color="#0b6e5a")
ax.set_xlabel("tiempo (min)")
ax.set_ylabel("dímero de coat protein (µM)")
ax.set_title("La temperatura es una palanca sobre la entrada de M4", loc="left")
ax.legend(frameon=False, title="cultivo")
fig.tight_layout()
fig.savefig("M3_temperatura.png", dpi=300, bbox_inches="tight")
plt.show()

## Uso del notebook

| Para probar... | Se modifica... |
|---|---|
| otro promotor | `alpha_tx_nM_min` |
| otro sitio de unión al ribosoma (RBS) | `k_tl_permin` |
| otro medio de cultivo | `t_double_min` |
| una inducción más tardía | `t_on` en `simulate_expression` |
| un dímero más débil | `Kd_dim_uM` |

**Cuando se disponga de los datos de RT-qPCR**, el siguiente paso es ajustar `alpha_tx_nM_min` y
`t_half_mrna_min` a la curva medida (Modo B). El diseño del muestreo es importante: un solo punto
final no permite identificar ambos parámetros por separado, y se necesitan unos 7 puntos en el
transitorio (2–180 min).